In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import ipywidgets as widgets

# Cargar los datos
datos = pd.read_csv("datos_ventas_platos.csv")

# Convertir el día de la semana a valores numéricos (0 = Lunes, 1 = Martes, ..., 6 = Domingo)
dias_semana_mapping = {'Lunes': 0, 'Martes': 1, 'Miércoles': 2, 'Jueves': 3, 'Viernes': 4, 'Sábado': 5, 'Domingo': 6}
datos['dia_semana'] = datos['dia_semana'].map(dias_semana_mapping)

# Lista de platos únicos
platos = datos["plato"].unique()

def plot_predictions(dia_pred):
    plt.figure(figsize=(15, len(platos) * 5))

    for i, plato in enumerate(platos):
        # Filtrar los datos por plato
        datos_plato = datos[datos["plato"] == plato]
        
        # Asegurarse de que hay datos suficientes para hacer predicciones
        if datos_plato.empty:
            print(f"No hay datos disponibles para el plato {plato}.")
            continue
        
        # Definir variables predictoras y objetivos
        X = datos_plato[["dia_semana"]]
        y_unidades = datos_plato["unidades_vendidas"]
        
        # Dividir los datos en conjuntos de entrenamiento y prueba
        X_train, X_test, y_unidades_train, y_unidades_test = train_test_split(X, y_unidades, test_size=0.2, random_state=42)
        
        # Entrenar el modelo: Bosque Aleatorio para Unidades
        modelo_unidades = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_unidades_train)
        
        # Definir el nuevo día para la predicción
        nuevo_dia = pd.DataFrame({"dia_semana": [dia_pred]})
        
        # Predicciones para el día elegido
        unidades_pred = modelo_unidades.predict(nuevo_dia)[0]
        
        # Cálculo de error cuadrático medio
        mse_unidades = mean_squared_error(y_unidades_test, modelo_unidades.predict(X_test))
        
        # Mostrar el error
        print(f"\nPlato {plato}:")
        print(f"  - Error cuadrático medio para Unidades (Bosque Aleatorio): {mse_unidades:.2f}")
        
        # Gráficos para el plato actual
        plt.subplot(len(platos), 1, i + 1)
        plt.plot(datos_plato["dia_semana"], datos_plato["unidades_vendidas"], label="Unidades históricas", marker='o')
        plt.scatter(dia_pred, unidades_pred, color="red", label="Predicción elegida")
        plt.text(dia_pred, unidades_pred, f"{unidades_pred:.0f}", color="red", ha='center', va='bottom')
        plt.xlabel("Día de la Semana")
        plt.ylabel("Unidades vendidas")
        plt.title(f"Predicción de Ventas - Plato {plato}")
        plt.xticks(ticks=list(dias_semana_mapping.values()), labels=list(dias_semana_mapping.keys()))
        plt.legend()

    # Ajustar el diseño y mostrar todos los gráficos
    plt.tight_layout()
    plt.show()

# Crear un slider para seleccionar el día de predicción
dia_slider = widgets.IntSlider(value=0, min=0, max=6, description='Día a predecir:')
widgets.interactive(plot_predictions, dia_pred=dia_slider)

interactive(children=(IntSlider(value=0, description='Día a predecir:', max=6), Output()), _dom_classes=('widg…